<a href="https://colab.research.google.com/github/BerkKilincer-dev/SoftIto/blob/main/llm-agents/03_langgraph_coklu_ajan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangGraph ve Çoklu Ajan Sistemleri

LangChain'in `create_agent`'ı tek bir hazır döngü sunar. Gerçek uygulamalar ise çoğu zaman daha fazla **kontrol** ister: dallanma, döngü sınırları, insan onayı, kaldığı yerden devam etme, birden fazla uzman ajanın iş birliği... **LangGraph**, LLM uygulamalarını **durum (state) taşıyan bir graf** olarak modellemeyi sağlar.

| Kavram | Anlamı |
|---|---|
| **State** | Graf boyunca taşınan ortak veri (ör. mesaj listesi, sayaçlar, ara sonuçlar) |
| **Node (düğüm)** | State'i alıp bir **güncelleme** döndüren Python fonksiyonu (LLM çağrısı, araç, iş mantığı) |
| **Edge (kenar)** | Hangi düğümden sonra hangisinin çalışacağı; **koşullu** kenarlar dallanma sağlar |
| **Reducer** | Bir state alanına gelen güncellemenin nasıl birleştirileceği (üzerine yaz / listeye ekle) |
| **Checkpointer** | Her adımdan sonra state'i kaydeder → hafıza, geri sarma, insan onayı |

| # | Bölüm |
|---|---|
| 1 | Kurulum (ücretsiz, yerel LLM: Ollama + Qwen2.5) |
| 2 | LLM'siz ilk graf: state, düğüm, koşullu kenar, döngü |
| 3 | Sıfırdan ReAct ajanı: `ToolNode` + `tools_condition` |
| 4 | Checkpointing: hafıza ve geçmiş durumlar |
| 5 | İnsan onayı (human-in-the-loop): `interrupt` ve `Command` |
| 6 | Çoklu ajan: **Supervisor** deseni |
| 7 | Yansıtma (reflection) döngüsü: yazar ⇄ eleştirmen |
| 8 | Paralel dağıtma (map-reduce): `Send` API |
| 9 | Çoklu ajan mimarileri ve tasarım ilkeleri |

> **Tamamen ücretsiz, API anahtarı yok.** `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU` seçin.

---
## 1) Kurulum

In [ ]:
!sudo apt-get install -y -qq zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q -U langgraph langchain langchain-ollama

In [ ]:
import subprocess, time, requests

ollama_sureci = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        requests.get("http://localhost:11434", timeout=1)
        print("Ollama sunucusu çalışıyor ✅")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)

MODEL = "qwen2.5:7b"
!ollama pull {MODEL}

In [ ]:
import operator
from typing import Annotated, Literal, TypedDict

from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatOllama(model=MODEL, temperature=0)
print(llm.invoke("Tek kelimeyle: LangGraph bir graf kütüphanesi mi?").content)

---
## 2) LLM'siz ilk graf

LangGraph'ın mekaniğini LLM olmadan, saf Python düğümleriyle görelim. Bir sayı tahmin oyunu: tahmin düğümü ikili arama ile tahmin yapar, kontrol düğümü sonucu değerlendirir, koşullu kenar ya döngüye geri döner ya da bitirir.

```
START ──► tahmin_et ──► kontrol_et ──┬──(doğru)──► END
              ▲                      │
              └─────(yanlış)─────────┘
```

In [ ]:
class OyunDurumu(TypedDict):
    hedef: int
    alt: int
    ust: int
    tahmin: int
    gecmis: Annotated[list[str], operator.add]     # reducer: her güncelleme listeye EKLENİR
    deneme: int


def tahmin_et(durum: OyunDurumu) -> dict:
    tahmin = (durum["alt"] + durum["ust"]) // 2
    return {"tahmin": tahmin, "deneme": durum["deneme"] + 1}     # sadece değişen alanları döndür


def kontrol_et(durum: OyunDurumu) -> dict:
    t, h = durum["tahmin"], durum["hedef"]
    if t < h:
        return {"alt": t + 1, "gecmis": [f"{t} → daha büyük"]}
    if t > h:
        return {"ust": t - 1, "gecmis": [f"{t} → daha küçük"]}
    return {"gecmis": [f"{t} → DOĞRU! 🎉"]}


def devam_mi(durum: OyunDurumu) -> Literal["tahmin_et", "__end__"]:
    return END if durum["tahmin"] == durum["hedef"] else "tahmin_et"


oyun = StateGraph(OyunDurumu)
oyun.add_node("tahmin_et", tahmin_et)
oyun.add_node("kontrol_et", kontrol_et)
oyun.add_edge(START, "tahmin_et")
oyun.add_edge("tahmin_et", "kontrol_et")
oyun.add_conditional_edges("kontrol_et", devam_mi)
oyun_grafi = oyun.compile()

sonuc = oyun_grafi.invoke({"hedef": 73, "alt": 1, "ust": 100, "tahmin": 0, "gecmis": [], "deneme": 0})
print("\n".join(sonuc["gecmis"]))
print(f"\n{sonuc['deneme']} denemede bulundu.")

In [ ]:
# Grafın yapısını Mermaid diyagramı olarak yazdır (mermaid.live'a yapıştırıp görebilirsiniz)
print(oyun_grafi.get_graph().draw_mermaid())

### `stream` ile adım adım izlemek

`stream(..., stream_mode="updates")` her düğümün çalışmasından sonra o düğümün döndürdüğü güncellemeyi verir. Uzun ajan çalışmalarını izlemek ve arayüzde ilerleme göstermek için kullanılır.

In [ ]:
for adim in oyun_grafi.stream({"hedef": 21, "alt": 1, "ust": 100, "tahmin": 0, "gecmis": [], "deneme": 0},
                              stream_mode="updates"):
    for dugum, guncelleme in adim.items():
        print(f"[{dugum:10s}] {guncelleme}")

---
## 3) Sıfırdan ReAct ajanı

**ReAct** (Reason + Act): model düşünür, bir araç çağırır, sonucu gözlemler, tekrar düşünür... Bunu LangGraph ile iki düğümlü bir döngü olarak kuruyoruz:

```
START ──► ajan (LLM) ──┬──(tool_calls var)──► araclar (ToolNode) ──┐
              ▲         │                                          │
              │         └──(tool_calls yok)──► END                 │
              └─────────────────────────────────────────────────────┘
```

- `MessagesState`: içinde `messages` alanı ve **`add_messages` reducer'ı** olan hazır state. Yeni mesajlar listeye eklenir.
- `ToolNode`: son AIMessage'daki araç çağrılarını çalıştırıp `ToolMessage` döndüren hazır düğüm.
- `tools_condition`: son mesajda araç çağrısı varsa `"tools"`, yoksa `END` döndüren hazır koşul.

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

URUNLER = {
    "laptop": {"fiyat": 42_000, "stok": 5},
    "monitör": {"fiyat": 8_500, "stok": 0},
    "klavye": {"fiyat": 1_750, "stok": 23},
    "mouse": {"fiyat": 650, "stok": 41},
}

@tool
def urun_bilgisi(urun: str) -> str:
    """Mağazadaki bir ürünün fiyatını (TL) ve stok adedini döndürür. Ürünler: laptop, monitör, klavye, mouse."""
    bilgi = URUNLER.get(urun.lower())
    return f"{urun}: {bilgi['fiyat']} TL, stok: {bilgi['stok']}" if bilgi else f"'{urun}' bulunamadı."

@tool
def hesapla(a: float, b: float, islem: Literal["topla", "cikar", "carp", "bol"]) -> float:
    """İki sayı üzerinde dört işlemden birini yapar."""
    return {"topla": a + b, "cikar": a - b, "carp": a * b, "bol": a / b if b else float("nan")}[islem]

@tool
def indirim_uygula(tutar: float, yuzde: float) -> float:
    """Bir tutara yüzde indirim uygular ve indirimli tutarı döndürür."""
    return round(tutar * (1 - yuzde / 100), 2)

magaza_araclari = [urun_bilgisi, hesapla, indirim_uygula]
llm_araclarla = llm.bind_tools(magaza_araclari)

SISTEM = SystemMessage("Sen bir elektronik mağazasının satış asistanısın. Fiyat ve stok bilgisini ASLA tahmin etme, "
                       "her zaman urun_bilgisi aracını kullan. Hesaplamalar için araçları kullan. Türkçe cevap ver.")

def ajan_dugumu(durum: MessagesState) -> dict:
    return {"messages": [llm_araclarla.invoke([SISTEM] + durum["messages"])]}

graf = StateGraph(MessagesState)
graf.add_node("ajan", ajan_dugumu)
graf.add_node("tools", ToolNode(magaza_araclari))
graf.add_edge(START, "ajan")
graf.add_conditional_edges("ajan", tools_condition)     # → "tools" veya END
graf.add_edge("tools", "ajan")
react_ajani = graf.compile()

In [ ]:
def calistir(graf, mesaj, config=None):
    for adim in graf.stream({"messages": [HumanMessage(mesaj)]}, config=config, stream_mode="updates"):
        for dugum, guncelleme in adim.items():
            if not guncelleme or "messages" not in guncelleme:
                continue
            for m in guncelleme["messages"]:
                if isinstance(m, AIMessage) and m.tool_calls:
                    for c in m.tool_calls:
                        print(f"  [{dugum}] 🔧 {c['name']}({c['args']})")
                elif isinstance(m, ToolMessage):
                    print(f"  [{dugum}] ↳ {m.content}")
                elif isinstance(m, AIMessage):
                    print(f"  [{dugum}] 💬 {m.content}")

calistir(react_ajani, "2 laptop ve 2 klavye alırsam toplam ne öderim? %10 indirim yaparsanız ne kadar olur?")
print()
calistir(react_ajani, "Monitör almak istiyorum, stokta var mı?")

---
## 4) Checkpointing: hafıza ve geçmiş durumlar

Grafı bir **checkpointer** ile derlediğimizde her düğümden sonra state kaydedilir. Bu üç güçlü özellik sağlar:

1. **Hafıza:** Aynı `thread_id` ile çağrılan graf önceki state'ten devam eder.
2. **Geçmiş / zaman yolculuğu:** Her ara state görüntülenebilir, istenen noktadan yeniden başlatılabilir.
3. **İnsan onayı:** Graf duraklatılıp sonra kaldığı yerden devam ettirilebilir (sonraki bölüm).

> `InMemorySaver` eğitim içindir. Üretimde `SqliteSaver`, `PostgresSaver` veya Redis tabanlı checkpointer'lar kullanılır.

In [ ]:
hafizali_ajan = graf.compile(checkpointer=InMemorySaver())
konusma = {"configurable": {"thread_id": "musteri-42"}}

calistir(hafizali_ajan, "Mouse'un fiyatı ne?", konusma)
print()
calistir(hafizali_ajan, "Ondan 3 tane alırsam toplam ne tutar?", konusma)      # "ondan" = mouse → hafıza gerekli

In [ ]:
durum = hafizali_ajan.get_state(konusma)
print(f"Thread'deki mesaj sayısı: {len(durum.values['messages'])}")
print(f"Sıradaki düğüm: {durum.next or '(bitti)'}\n")

print("Kaydedilmiş geçmiş durumlar (en yeniden eskiye):")
for i, anlik in enumerate(hafizali_ajan.get_state_history(konusma)):
    son = anlik.values["messages"][-1] if anlik.values.get("messages") else None
    print(f"  {i:2d}. adım={anlik.metadata.get('step')}, sıradaki={anlik.next}, "
          f"son mesaj={type(son).__name__ if son else '-'}")

---
## 5) İnsan onayı (Human-in-the-loop)

Ajanlar gerçek dünyada işlem yapmaya başladığında (e-posta gönderme, ödeme, veri silme), kritik adımlardan önce bir insanın onayı gerekir. LangGraph'ta bir düğüm veya araç içinde `interrupt(...)` çağrıldığında:

1. Graf **duraklar**, state checkpointer'a kaydedilir ve `interrupt`'a verilen bilgi çağırana döner.
2. İnsan kararını verir.
3. `graph.invoke(Command(resume=karar), config)` ile graf **tam kaldığı yerden** devam eder; `interrupt(...)` bu kez kararı döndürür.

> `interrupt` checkpointer gerektirir, çünkü graf saniyeler, saatler hatta günler sonra devam ettirilebilir.

In [ ]:
from langgraph.types import interrupt, Command

@tool
def siparis_olustur(urun: str, adet: int) -> str:
    """Müşteri için sipariş oluşturur ve kartından ödemeyi çeker. Bu işlem geri alınamaz."""
    karar = interrupt({"islem": "siparis_olustur", "urun": urun, "adet": adet,
                       "soru": f"{adet} adet {urun} için sipariş oluşturulup ödeme çekilsin mi? (evet/hayır)"})
    if str(karar).strip().lower() != "evet":
        return "Sipariş insan onayı alınamadığı için İPTAL edildi."
    return f"✅ Sipariş oluşturuldu: {adet} adet {urun}. Sipariş no: SP-{abs(hash((urun, adet))) % 100000:05d}"

onayli_araclar = magaza_araclari + [siparis_olustur]
llm_onayli = llm.bind_tools(onayli_araclar)

def onayli_ajan_dugumu(durum: MessagesState) -> dict:
    return {"messages": [llm_onayli.invoke([SISTEM] + durum["messages"])]}

g = StateGraph(MessagesState)
g.add_node("ajan", onayli_ajan_dugumu)
g.add_node("tools", ToolNode(onayli_araclar))
g.add_edge(START, "ajan")
g.add_conditional_edges("ajan", tools_condition)
g.add_edge("tools", "ajan")
onayli_ajan = g.compile(checkpointer=InMemorySaver())

In [ ]:
config = {"configurable": {"thread_id": "siparis-1"}}
sonuc = onayli_ajan.invoke({"messages": [HumanMessage("2 adet klavye siparişi oluştur lütfen.")]}, config)

if "__interrupt__" in sonuc:
    bekleyen = sonuc["__interrupt__"][0].value
    print("⏸️  Graf duraklatıldı, insan onayı bekleniyor:")
    print(f"    {bekleyen['soru']}")
    print(f"    Sıradaki düğüm: {onayli_ajan.get_state(config).next}")

In [ ]:
# İnsan kararını ver ve kaldığı yerden devam et (reddetmek için "hayır" yazın)
insan_karari = "evet"
sonuc = onayli_ajan.invoke(Command(resume=insan_karari), config)
print(f"👤 Karar: {insan_karari}")
print(f"💬 {sonuc['messages'][-1].content}")

---
## 6) Çoklu ajan: Supervisor (yönetici) deseni

Tek bir ajana çok fazla araç ve sorumluluk vermek performansı düşürür: model yanlış aracı seçer, uzun sistem istemleri karışır, bağlam şişer. Çözüm: işi **uzmanlaşmış ajanlara** bölmek.

**Supervisor deseninde** bir yönetici ajan, gelen görevi analiz eder ve sıradaki adımı hangi uzmanın yapacağına karar verir. Uzman işini bitirince kontrol yöneticiye döner. Yönetici işin bittiğine karar verene kadar bu sürer.

```
                       ┌─────────────┐
           ┌──────────►│  supervisor │◄──────────┐
           │           └──────┬──────┘           │
           │      ┌───────────┼───────────┐      │
           │      ▼           ▼           ▼      │
           │ araştırmacı   analist      yazar    │
           │ (bilgi aracı) (hesap aracı) (araçsız)│
           └──────┴───────────┴───────────┴──────┘
                              │
                          BITIR → END
```

In [ ]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field

SIRKET_BILGI = {
    "novacrm": "NovaCRM paketleri: Başlangıç kullanıcı başına aylık 290 TL (en fazla 10 kullanıcı), "
               "Profesyonel kullanıcı başına aylık 590 TL (en fazla 100 kullanıcı). Yıllık ödemede %20 indirim.",
    "destek": "Başlangıç: hafta içi e-posta desteği. Profesyonel: canlı sohbet desteği eklenir. Kurumsal: 7/24 telefon.",
    "entegrasyon": "NovaCRM; Gmail, Outlook ve Slack ile entegre çalışır. REST API sadece Profesyonel ve Kurumsal pakette.",
}

@tool
def sirket_bilgisi_ara(konu: str) -> str:
    """NovaCRM ürünüyle ilgili şirket bilgi tabanında arama yapar. Konular: novacrm (fiyat/paketler), destek, entegrasyon."""
    sonuclar = [metin for anahtar, metin in SIRKET_BILGI.items() if anahtar in konu.lower() or konu.lower() in metin.lower()]
    return "\n".join(sonuclar) if sonuclar else "\n".join(SIRKET_BILGI.values())

@tool
def hesap(ifade: str) -> str:
    """Basit bir aritmetik ifadeyi hesaplar, örnek: '590 * 25 * 12 * 0.8'. Sadece sayılar ve + - * / ( ) kullan."""
    if not set(ifade) <= set("0123456789+-*/(). "):
        return "Hata: ifade sadece sayı ve + - * / ( ) içermeli."
    return str(round(eval(ifade), 2))     # karakter kümesi kısıtlandığı için güvenli

arastirmaci = create_agent(llm, tools=[sirket_bilgisi_ara],
    system_prompt="Sen bir araştırmacısın. Görev için gereken ürün bilgilerini sirket_bilgisi_ara aracıyla bul ve "
                  "bulduğun gerçekleri madde madde, kısaca raporla. Hesaplama YAPMA.")
analist = create_agent(llm, tools=[hesap],
    system_prompt="Sen bir finans analistisin. Konuşmadaki bilgileri kullanarak gereken hesaplamaları hesap aracıyla yap. "
                  "Her hesaplamayı ve sonucunu açıkça yaz. Bilgi uydurma.")
yazar = create_agent(llm, tools=[],
    system_prompt="Sen bir satış iletişimi uzmanısın. Konuşmadaki araştırma ve hesaplama sonuçlarını kullanarak müşteriye "
                  "gönderilecek kısa, profesyonel ve Türkçe bir teklif e-postası yaz. Yeni rakam uydurma.")
UZMANLAR = {"arastirmaci": arastirmaci, "analist": analist, "yazar": yazar}

### Supervisor: yapılandırılmış çıktı ile yönlendirme

Yönetici, serbest metin yerine **şemaya uygun bir karar** döndürür. Böylece yönlendirme güvenilir şekilde ayrıştırılır. Sonsuz döngülere karşı bir **adım sınırı** da koyuyoruz.

In [ ]:
class Yonlendirme(BaseModel):
    gerekce: str = Field(description="Kararın kısa gerekçesi")
    siradaki: Literal["arastirmaci", "analist", "yazar", "BITIR"] = Field(description="Sıradaki adımı yapacak uzman")

SUPERVISOR_ISTEMI = """Sen bir ekip yöneticisisin. Ekibin:
- arastirmaci: şirket bilgi tabanından ürün/fiyat bilgisi bulur
- analist: hesaplama yapar
- yazar: müşteriye gidecek son metni yazar
Konuşmaya bakarak sıradaki adımı kimin yapacağına karar ver.
Kurallar: Bilgi yoksa önce arastirmaci. Hesap gerekiyorsa ve henüz yapılmadıysa analist.
Tüm bilgi ve hesaplar hazırsa yazar. Yazar e-postayı yazdıysa BITIR. Aynı uzmanı gereksiz yere tekrar çağırma."""

yonetici_llm = llm.with_structured_output(Yonlendirme)
MAKS_ADIM = 8

class EkipDurumu(MessagesState):
    siradaki: str
    adim: int

def supervisor(durum: EkipDurumu) -> dict:
    if durum.get("adim", 0) >= MAKS_ADIM:
        return {"siradaki": "BITIR"}
    karar = yonetici_llm.invoke([SystemMessage(SUPERVISOR_ISTEMI)] + durum["messages"])
    print(f"🧭 supervisor → {karar.siradaki}  ({karar.gerekce})")
    return {"siradaki": karar.siradaki, "adim": durum.get("adim", 0) + 1}

def uzman_dugumu(isim):
    def calistir(durum: EkipDurumu) -> dict:
        sonuc = UZMANLAR[isim].invoke({"messages": durum["messages"]})
        cikti = sonuc["messages"][-1].content
        print(f"\n👷 {isim}:\n{cikti}\n")
        return {"messages": [AIMessage(content=cikti, name=isim)]}
    return calistir

ekip = StateGraph(EkipDurumu)
ekip.add_node("supervisor", supervisor)
for isim in UZMANLAR:
    ekip.add_node(isim, uzman_dugumu(isim))
    ekip.add_edge(isim, "supervisor")                 # her uzman işini bitirince yöneticiye döner
ekip.add_edge(START, "supervisor")
ekip.add_conditional_edges("supervisor", lambda d: END if d["siradaki"] == "BITIR" else d["siradaki"])
ekip_grafi = ekip.compile()

print(ekip_grafi.get_graph().draw_mermaid())

In [ ]:
gorev = ("25 kişilik bir satış ekibi için NovaCRM Profesyonel paketiyle ilgili teklif hazırla. "
         "Yıllık ödeme yaparlarsa toplam yıllık maliyeti hesapla ve REST API desteği olup olmadığını belirt.")
sonuc = ekip_grafi.invoke({"messages": [HumanMessage(gorev)], "adim": 0}, {"recursion_limit": 30})
print("═" * 100)
print("NİHAİ ÇIKTI:\n")
print(sonuc["messages"][-1].content)

---
## 7) Yansıtma (Reflection) döngüsü: yazar ⇄ eleştirmen

Bir ajanın çıktısını başka bir ajanın **eleştirmesi** ve ilkinin buna göre düzeltmesi, kaliteyi belirgin şekilde artırabilir. Döngüyü bir **revizyon sınırı** ile koruyoruz.

```
START ──► yazar ──► elestirmen ──┬──(ONAY veya sınır doldu)──► END
             ▲                   │
             └──(geri bildirim)──┘
```

In [ ]:
class YaziDurumu(TypedDict):
    konu: str
    taslak: str
    geri_bildirim: str
    revizyon: int
    onaylandi: bool

class Degerlendirme(BaseModel):
    onay: bool = Field(description="Metin tüm kriterleri karşılıyorsa true")
    geri_bildirim: str = Field(description="Onay yoksa somut, maddeli düzeltme önerileri")

def yazar_dugumu(d: YaziDurumu) -> dict:
    istem = f"'{d['konu']}' hakkında lise öğrencileri için en fazla 80 kelimelik bir paragraf yaz."
    if d.get("geri_bildirim"):
        istem += f"\n\nÖnceki taslağın:\n{d['taslak']}\n\nEleştirmenin geri bildirimi:\n{d['geri_bildirim']}\n\nBuna göre düzelt."
    taslak = llm.invoke(istem).content
    print(f"✍️  Taslak (revizyon {d.get('revizyon', 0)}):\n{taslak}\n")
    return {"taslak": taslak, "revizyon": d.get("revizyon", 0) + 1}

elestirmen_llm = llm.with_structured_output(Degerlendirme)

def elestirmen_dugumu(d: YaziDurumu) -> dict:
    karar = elestirmen_llm.invoke(
        "Aşağıdaki metni şu kriterlere göre değerlendir: (1) en fazla 80 kelime, (2) teknik terimler açıklanmış, "
        f"(3) somut bir günlük hayat örneği içeriyor, (4) akıcı Türkçe.\n\nMetin:\n{d['taslak']}")
    print(f"🧐 Eleştirmen: {'ONAY ✅' if karar.onay else 'DÜZELT'} — {karar.geri_bildirim}\n")
    return {"onaylandi": karar.onay, "geri_bildirim": karar.geri_bildirim}

yg = StateGraph(YaziDurumu)
yg.add_node("yazar", yazar_dugumu)
yg.add_node("elestirmen", elestirmen_dugumu)
yg.add_edge(START, "yazar")
yg.add_edge("yazar", "elestirmen")
yg.add_conditional_edges("elestirmen", lambda d: END if d["onaylandi"] or d["revizyon"] >= 3 else "yazar")
yansitma = yg.compile()

son = yansitma.invoke({"konu": "makine öğrenmesinde overfitting", "taslak": "", "geri_bildirim": "",
                       "revizyon": 0, "onaylandi": False})
print("═" * 80, f"\nSON METİN ({son['revizyon']} revizyon):\n{son['taslak']}")

---
## 8) Paralel dağıtma (map-reduce): `Send` API

Bazen aynı işi **değişken sayıda** parçaya paralel uygulamak gerekir: N dokümanı ayrı ayrı özetle, sonra özetleri birleştir. `Send(düğüm, girdi)` nesneleri döndüren bir koşullu kenar, aynı düğümün birden fazla kopyasını **paralel** çalıştırır. Sonuçlar bir reducer (`operator.add`) ile tek listede toplanır.

```
           ┌──► ozetle(doküman 1) ──┐
START ─────┼──► ozetle(doküman 2) ──┼──► birlestir ──► END
           └──► ozetle(doküman 3) ──┘
```

In [ ]:
from langgraph.types import Send

class RaporDurumu(TypedDict):
    dokumanlar: list[str]
    ozetler: Annotated[list[str], operator.add]       # paralel kopyaların çıktıları burada birleşir
    rapor: str

class TekDokuman(TypedDict):
    dokuman: str

def dagit(d: RaporDurumu):
    return [Send("ozetle", {"dokuman": dok}) for dok in d["dokumanlar"]]

def ozetle(d: TekDokuman) -> dict:
    return {"ozetler": [llm.invoke(f"Şu müşteri geri bildirimini tek cümleyle özetle:\n{d['dokuman']}").content]}

def birlestir(d: RaporDurumu) -> dict:
    liste = "\n".join(f"- {o}" for o in d["ozetler"])
    rapor = llm.invoke("Aşağıdaki müşteri geri bildirimi özetlerinden yönetime 3 maddelik bir aksiyon raporu çıkar:\n" + liste).content
    return {"rapor": rapor}

mr = StateGraph(RaporDurumu)
mr.add_node("ozetle", ozetle)
mr.add_node("birlestir", birlestir)
mr.add_conditional_edges(START, dagit, ["ozetle"])
mr.add_edge("ozetle", "birlestir")
mr.add_edge("birlestir", END)
map_reduce = mr.compile()

geri_bildirimler = [
    "Uygulama çok yavaş açılıyor, özellikle sabahları raporlar sayfası 30 saniye yükleniyor.",
    "Destek ekibi çok ilgiliydi, sorunumu 10 dakikada çözdüler. Teşekkürler!",
    "Mobil uygulamada müşteri kartını düzenleyemiyorum, kaydet butonu çalışmıyor.",
    "Fiyatlar rakiplere göre biraz yüksek ama özellikler buna değiyor.",
    "Slack entegrasyonu bildirimleri bazen iki kez gönderiyor, ekip rahatsız oluyor.",
]
sonuc = map_reduce.invoke({"dokumanlar": geri_bildirimler, "ozetler": [], "rapor": ""})
print("Özetler:"); print("\n".join(f"  • {o}" for o in sonuc["ozetler"]))
print("\nRapor:\n" + sonuc["rapor"])

---
## 9) Çoklu ajan mimarileri ve tasarım ilkeleri

| Mimari | Nasıl çalışır? | Ne zaman? |
|---|---|---|
| **Supervisor** | Merkezi yönetici, işi uzmanlara dağıtır | Görev adımları önceden net değil, uzmanlıklar ayrışık |
| **Hiyerarşik** | Yöneticilerin yöneticisi; her alt ekibin kendi supervisor'ı var | Çok sayıda ajan, büyük sistemler |
| **Ağ / Swarm (handoff)** | Ajanlar kontrolü doğrudan birbirine devreder | Müşteri hizmetleri gibi konuşmanın bir uzmandan diğerine geçtiği akışlar |
| **Sıralı hat (pipeline)** | Sabit sıra: planla → yap → kontrol et | Adımlar baştan belli; en öngörülebilir ve ucuz yapı |
| **Yansıtma (reflection)** | Üretici ⇄ eleştirmen döngüsü | Kalite kritik, değerlendirme kriterleri net |
| **Map-reduce** | Aynı işi N parçaya paralel uygula, sonra birleştir | Çok sayıda bağımsız alt görev (doküman, kayıt, dosya) |

**Tasarım ilkeleri**
- **Önce basit başla.** Tek bir LLM çağrısı veya sabit bir pipeline işi görüyorsa ajan kurma. Ajanlar daha pahalı, daha yavaş ve daha az öngörülebilirdir.
- **Her ajanın tek bir net sorumluluğu ve az sayıda aracı olsun.** Araç sayısı arttıkça yanlış araç seçimi artar.
- **Döngülere sınır koy:** `recursion_limit`, adım sayacı, revizyon sınırı.
- **Kararları yapılandırılmış çıktıyla al** (yönlendirme, onay); serbest metni ayrıştırmaya çalışma.
- **Geri alınamaz işlemlerden önce `interrupt` ile insan onayı** iste; checkpointer ile durumu kalıcı tut.
- **Gözlemlenebilirlik:** `stream` ile adımları izle; üretimde LangSmith, Langfuse gibi izleme araçları kullan.
- **Güvenlik:** Araç çıktıları ve getirilen dokümanlar güvenilmeyen veridir; dolaylı prompt enjeksiyonuna karşı ajan yetkilerini sınırla (→ `transformers-llm/09_llm_guvenligi.md`).

## Özet

| Kavram | LangGraph karşılığı |
|---|---|
| Durum | `TypedDict` / `MessagesState` + `Annotated[..., reducer]` |
| Düğüm | `graph.add_node("ad", fonksiyon)` → state güncellemesi döndürür |
| Kenar / dallanma | `add_edge`, `add_conditional_edges` |
| Araç çalıştırma | `ToolNode` + `tools_condition` |
| Hafıza / geçmiş | `compile(checkpointer=InMemorySaver())`, `thread_id`, `get_state`, `get_state_history` |
| İnsan onayı | `interrupt(...)` + `Command(resume=...)` |
| Paralellik | `Send(düğüm, girdi)` + `operator.add` reducer |
| Çoklu ajan | Supervisor düğümü + uzman ajan düğümleri + koşullu yönlendirme |